# Pretraining Dataset Preparation — U-Net Building Detector

This notebook runs the complete dataset preparation pipeline for the U-Net building detector pretraining. Starting from Planet quads (8 bands) and ALKIS building footprints (EPSG:25832), it produces a 512×512 tile dataset ready for training, with train/val/test splits and per-band normalisation statistics.

**Inputs:**
- `data/raw/planet_quads/` — Planet `.tif` files (8 spectral bands + alpha band 9)
- `data/raw/alkis/gebaude.gpkg` — building footprints (ALKIS, EPSG:25832)

**Outputs in `data/pretraining/`:**
- `mosaic.tif`, `mosaic.grid.json` — reprojected mosaic and its grid JSON
- `building_mask.tif` — binary building mask aligned to the mosaic
- `tiles/images/`, `tiles/masks/` — 512×512 image and mask tiles
- `tiles/splits/train.txt`, `val.txt`, `test.txt` — splits by tile ID
- `stats/norm_stats_building_8b.json` — per-band mean and standard deviation

## 0. Setup

Imports, environment configuration, and definition of **all pipeline parameters in a single place**.
Modify the values in this section before running the rest of the notebook.
At the end of the section the existence of input files is verified; if any are missing, a clear error message is printed.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from IPython.display import display

# Add src/ to the path to import project modules
project_root = Path(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(str(project_root / "src"))

from config import DATA_RAW, DATA_DIR
from build_pretrain_dataset import (
    build_mosaic,
    rasterize_mask,
    tile_dataset,
    make_splits,
    compute_norm,
)

# -------------------------------------------------------------------------
# Configurable parameters — modify here and only here
# -------------------------------------------------------------------------

# Inputs
quads_dir       = DATA_RAW / "planet_quads"              # directory containing Planet .tif files
footprints_path = DATA_RAW / "alkis" / "gebaude.gpkg"    # ALKIS footprints
alkis_layer     = None                                   # GPKG layer name (None = auto)

# Output root
out_dir = DATA_DIR / "pretraining"

# Derived paths (do not modify)
out_tif        = out_dir / "mosaic.tif"
out_grid_json  = out_dir / "mosaic.grid.json"
out_mask       = out_dir / "building_mask.tif"
tiles_dir      = out_dir / "tiles"
images_dir     = tiles_dir / "images"
masks_dir      = tiles_dir / "masks"
splits_dir     = tiles_dir / "splits"
stats_dir      = out_dir / "stats"
stats_name     = "norm_stats_building_8b.json"
out_stats_json = stats_dir / stats_name

# Mosaic parameters
epsg  = 25832
res   = 3.0                    # resolution in metres
bands = list(range(1, 9))      # bands 1–8 (excludes alpha band 9)

# Tiling parameters
tile_size = 512
stride    = 512
pos_min   = 0.01               # minimum fraction of positive pixels to keep a tile
id_prefix = "tile"

# Split parameters
train_frac = 0.70
val_frac   = 0.15
seed       = 42

# -------------------------------------------------------------------------
# Input verification
# -------------------------------------------------------------------------
missing = []
if not quads_dir.exists():
    missing.append(f"ERROR: quads directory not found: {quads_dir}")
elif not list(quads_dir.glob("*.tif")):
    missing.append(f"ERROR: no .tif files found in: {quads_dir}")
if not footprints_path.exists():
    missing.append(f"ERROR: footprints file not found: {footprints_path}")

if missing:
    for msg in missing:
        print(msg)
    raise FileNotFoundError("Incomplete inputs. Check the paths in the Setup section.")
else:
    print("Inputs verified successfully.")
    print(f"  quads_dir:       {quads_dir}")
    print(f"  footprints_path: {footprints_path}")
    print(f"  out_dir:         {out_dir}")

## 1. Input Inspection

Before running the pipeline, we inspect the input data to confirm it is compatible with the configured parameters. This section can be run independently once the inputs exist.

### 1.1 Planet Quads

Lists the quads available in `planet_quads/` and opens the first one with rasterio to print its properties: native CRS, number of bands (expected: 9 including alpha), resolution, and dimensions.

In [ ]:
quad_files = sorted(quads_dir.glob("*.tif"))
print(f"Available quads: {len(quad_files)}")
print(f"First file:      {quad_files[0].name}")

# Inspect the first quad with rasterio
with rasterio.open(quad_files[0]) as src:
    print("\n--- Quad properties ---")
    print(f"  CRS:         {src.crs}")
    print(f"  N bands:     {src.count}")
    print(f"  Resolution:  {src.res[0]:.4f} x {src.res[1]:.4f}  (CRS units)")
    print(f"  Size:        {src.width} x {src.height} px")
    print(f"  Dtype:       {src.dtypes[0]}")
    print(f"  Bounds:      {src.bounds}")

### 1.2 Building Footprints (ALKIS)

Loads the GeoPackage containing the footprints and displays the CRS, number of geometries, and spatial extent. The CRS should be EPSG:25832; if it differs, `rasterize_mask` will reproject automatically.

In [ ]:
# Inspect the building footprints
gdf = gpd.read_file(footprints_path, layer=alkis_layer)
print("--- Footprint properties ---")
print(f"  CRS:          {gdf.crs}")
print(f"  N geometries: {len(gdf)}")
print(f"  Bounds:       {gdf.total_bounds.tolist()}")
display(gdf.head(3))

## 2. Mosaic Construction (`build_mosaic`)

Reprojects and merges all Planet quads into a single GeoTIFF in the target CRS (EPSG:25832) at the configured resolution (3 m/px). Only the spectral bands 1–8 are included; the alpha band (9) is discarded.

Outputs:
- `mosaic.tif` — mosaic aligned to the target grid
- `mosaic.grid.json` — dictionary with transform, dimensions, and bounds; this is the contract between pipeline stages

**Note:** this stage may take several minutes depending on the number and size of the quads.

In [ ]:
grid = build_mosaic(
    quads_dir=quads_dir,
    out_tif=out_tif,
    out_grid_json=out_grid_json,
    epsg=epsg,
    res=res,
    bands=bands,
)

# Verify the file was created and print its size on disk
size_mb = out_tif.stat().st_size / 1024**2
print(f"\nmosaic.tif created: {out_tif}")
print(f"Size on disk:       {size_mb:.1f} MB")

print("\nGrid JSON:")
print(json.dumps(grid, indent=2))

## 3. Building Mask Rasterisation (`rasterize_mask`)

Converts the ALKIS vector footprints into a binary mask (uint8) aligned pixel-by-pixel with the mosaic. Pixels with value 1 correspond to buildings; value 0 to background.

The visualisation shows the full mask along with the percentage of positive pixels, giving an indication of building density in the study area. A value between 5% and 25% is typical in dense urban areas.

In [ ]:
# If this section is run independently, load the grid from disk
if "grid" not in dir():
    grid = json.loads(out_grid_json.read_text(encoding="utf-8"))

rasterize_mask(
    footprints_path=footprints_path,
    grid=grid,
    out_mask=out_mask,
    layer=alkis_layer,
)

# Read and visualise the mask
with rasterio.open(out_mask) as src:
    mask_arr = src.read(1)  # shape: (H, W), dtype: uint8

pos_pct = 100.0 * mask_arr.sum() / mask_arr.size

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mask_arr, cmap="gray", interpolation="nearest", aspect="auto")
ax.set_title(
    f"Building mask — {pos_pct:.2f}% positive pixels\n"
    f"({mask_arr.shape[1]} x {mask_arr.shape[0]} px)"
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Tile Generation (`tile_dataset`)

Scans the mosaic with a sliding window of 512×512 pixels (stride=512, no overlap). Only tiles whose mask has at least `pos_min` = 1% positive pixels are kept, ensuring building signal in every training sample.

Image tiles are saved as multi-band GeoTIFFs in `tiles/images/` and binary masks in `tiles/masks/`.

In [ ]:
tile_ids = tile_dataset(
    img_path=out_tif,
    msk_path=out_mask,
    out_images_dir=images_dir,
    out_masks_dir=masks_dir,
    tile_size=tile_size,
    stride=stride,
    pos_min=pos_min,
    id_prefix=id_prefix,
)

print(f"Total tiles generated: {len(tile_ids)}")

### 4.1 Sample Visualisation

Four random tiles are selected (fixed seed for reproducibility) and displayed as an RGB composite (bands 4, 3, 2) with p2/p98 normalisation, alongside the building mask overlaid in semi-transparent red. Tiles are read directly from disk, so this cell can be re-run without regenerating the tiles.

In [ ]:
# If this section is run independently, recover tile_ids from disk
if "tile_ids" not in dir():
    tile_ids = [p.stem for p in sorted(images_dir.glob("*.tif"))]
    print(f"Tiles loaded from disk: {len(tile_ids)}")


def stretch_rgb(arr_chw, p_low=2, p_high=98):
    """Normalises each channel by p2/p98 percentiles to the range [0, 1]."""
    n_ch = arr_chw.shape[0]
    out = np.zeros_like(arr_chw, dtype=np.float32)
    for i in range(n_ch):
        ch = arr_chw[i].astype(np.float32)
        lo, hi = np.percentile(ch, [p_low, p_high])
        hi = hi if hi > lo else lo + 1e-6
        out[i] = np.clip((ch - lo) / (hi - lo), 0.0, 1.0)
    return np.transpose(out, (1, 2, 0))  # (H, W, C)


# Select 4 random tiles to visualise
random.seed(seed)
sample_ids = random.sample(tile_ids, min(4, len(tile_ids)))

fig, axes = plt.subplots(len(sample_ids), 2, figsize=(10, 4 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = [axes]  # ensure list of rows

for row_axes, tid in zip(axes, sample_ids):
    img_tile_path = images_dir / f"{tid}.tif"
    msk_tile_path = masks_dir  / f"{tid}_mask.tif"

    with rasterio.open(img_tile_path) as si:
        img = si.read()   # shape: (C, H, W)
    with rasterio.open(msk_tile_path) as sm:
        msk = sm.read(1)  # shape: (H, W)

    # RGB composite: bands 4, 3, 2 (0-based indices: 3, 2, 1)
    rgb = stretch_rgb(img[[3, 2, 1], :, :])

    # Left panel: RGB image
    row_axes[0].imshow(rgb)
    row_axes[0].set_title(f"{tid}\nRGB (B4-B3-B2)")
    row_axes[0].axis("off")

    # Right panel: RGB image + overlaid mask
    row_axes[1].imshow(rgb)
    row_axes[1].imshow(
        np.ma.masked_where(msk == 0, msk),
        cmap="Reds", alpha=0.5, vmin=0, vmax=1
    )
    pos_pct_tile = 100.0 * msk.sum() / msk.size
    row_axes[1].set_title(f"Overlaid mask\n({pos_pct_tile:.1f}% positive)")
    row_axes[1].axis("off")

plt.suptitle("Sample generated tiles (4 random)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Split Generation (`make_splits`)

Randomly divides the tile IDs into three disjoint subsets:
- **train** (70%) — used for model training
- **val** (15%) — used to monitor training and select hyperparameters
- **test** (15%) — reserved for final evaluation (do not use during development)

The IDs for each split are saved in `.txt` files inside `tiles/splits/`. The seed `seed=42` ensures reproducibility.

In [ ]:
# If this section is run independently, recover tile_ids from disk
if "tile_ids" not in dir():
    tile_ids = [p.stem for p in sorted(images_dir.glob("*.tif"))]

splits = make_splits(
    tile_ids=tile_ids,
    out_splits_dir=splits_dir,
    train_frac=train_frac,
    val_frac=val_frac,
    seed=seed,
)

# Count table per split
total = sum(len(v) for v in splits.values())
print("\nTile distribution by split:")
print(f"{'Split':<10} {'N tiles':>8}   {'%':>6}")
print("-" * 30)
for split_name, ids in splits.items():
    pct = 100.0 * len(ids) / total if total > 0 else 0.0
    print(f"{split_name:<10} {len(ids):>8}   {pct:>5.1f}%")
print("-" * 30)
print(f"{'TOTAL':<10} {total:>8}   100.0%")

## 6. Normalisation Statistics (`compute_norm`)

Computes the per-band mean and standard deviation using **only the train tiles**, avoiding any information leakage from the validation/test sets. The statistics are saved to a JSON that will be consumed by `pretrain_unet.py` during training.

The two-pass algorithm (mean of per-tile means) is a memory-efficient approximation; it is exact when all tiles have the same number of pixels, which is the case here (all are 512×512).

In [ ]:
# If this section is run independently, load splits from disk
if "splits" not in dir():
    splits = {}
    for split_name in ("train", "val", "test"):
        txt = splits_dir / f"{split_name}.txt"
        splits[split_name] = txt.read_text(encoding="utf-8").splitlines() if txt.exists() else []

norm_stats = compute_norm(
    images_dir=images_dir,
    train_ids=splits["train"],
    out_json=out_stats_json,
)

print("\nNormalisation statistics (first 3 values):")
print(f"  mean: {[round(v, 2) for v in norm_stats['mean']]}")
print(f"  std:  {[round(v, 2) for v in norm_stats['std']]}")

### 6.1 Statistics Visualisation

Grouped bar chart with the per-band mean and standard deviation. Allows rapid identification of brightness differences between bands and detection of potential anomalies (e.g., a band with a very low mean would indicate a possible un-excluded alpha band).

In [ ]:
# If this section is run independently, load stats from disk
if "norm_stats" not in dir():
    norm_stats = json.loads(out_stats_json.read_text(encoding="utf-8"))

means = np.array(norm_stats["mean"])
stds  = np.array(norm_stats["std"])
n_bands_stat = len(means)
band_labels  = [f"B{i+1}" for i in range(n_bands_stat)]
x     = np.arange(n_bands_stat)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
bars_mean = ax.bar(x - width / 2, means, width, label="Mean", color="steelblue")
bars_std  = ax.bar(x + width / 2, stds,  width, label="Std",  color="coral")

# Annotate values above each bar
for bar in bars_mean:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8
    )
for bar in bars_std:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8
    )

ax.set_xticks(x)
ax.set_xticklabels(band_labels)
ax.set_xlabel("Band")
ax.set_ylabel("Pixel value")
ax.set_title("Per-band normalisation statistics (computed on train tiles)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Final Summary

Verifies that all pipeline outputs exist on disk and prints a summary table with the key metrics of the generated dataset. If any output is missing, it is flagged with `[MISSING]` to help identify the failed stage.

In [ ]:
# Recover splits if not in memory (partial notebook execution)
if "splits" not in dir():
    splits = {}
    for split_name in ("train", "val", "test"):
        txt = splits_dir / f"{split_name}.txt"
        splits[split_name] = txt.read_text(encoding="utf-8").splitlines() if txt.exists() else []

# Verify existence of all expected outputs
expected_outputs = {
    "mosaic.tif":                 out_tif,
    "mosaic.grid.json":           out_grid_json,
    "building_mask.tif":          out_mask,
    "tiles/images/ (directory)":  images_dir,
    "tiles/masks/  (directory)":  masks_dir,
    "splits/train.txt":           splits_dir / "train.txt",
    "splits/val.txt":             splits_dir / "val.txt",
    "splits/test.txt":            splits_dir / "test.txt",
    "norm_stats JSON":             out_stats_json,
}

print("Output verification:")
all_ok = True
for label, path in expected_outputs.items():
    ok = path.exists()
    status = "OK     " if ok else "MISSING"
    if not ok:
        all_ok = False
    print(f"  [{status}]  {label}")

print()
if all_ok:
    print("All outputs generated successfully.")
else:
    print("WARNING: some outputs were not found. Check the previous stages.")

# Dataset summary table
n_train = len(splits.get("train", []))
n_val   = len(splits.get("val",   []))
n_test  = len(splits.get("test",  []))
n_total = n_train + n_val + n_test

def _pct(n, total):
    return f"({100 * n / total:.1f}%)" if total > 0 else ""

print("\n" + "=" * 48)
print(" PRETRAINING DATASET SUMMARY")
print("=" * 48)
print(f"  Total tiles:        {n_total}")
print(f"  Train tiles:        {n_train:>6}  {_pct(n_train, n_total)}")
print(f"  Val tiles:          {n_val:>6}  {_pct(n_val,   n_total)}")
print(f"  Test tiles:         {n_test:>6}  {_pct(n_test,  n_total)}")
print(f"  Tile size:          {tile_size} x {tile_size} px")
print(f"  N bands:            {len(bands)}")
print(f"  Resolution:         {res} m/px")
print(f"  Stats JSON:         {out_stats_json}")
print("=" * 48)